# Exercise 3 · Score a prompt

You will score two versions of a prompt (the instructions a model gets), field by field, and keep the better one.

**Goal:** Measure how often each field of a model's replies is right.

**Why it matters:** A few replies that look right say little about the ones you did not read. Labels, answers a person checked, show how often each field is wrong, and whether a change helped.

**From the slides:** Slide 22, "Reading by eye fails in two ways; a labelled set fixes both"; Slide 23, "Start by looking, not by measuring"; Slide 24, "Score each field, set the bar, and run it on every change"; Slide 27, "The two measures separate the two ways an extractor can be wrong".

**Before you start:** You can write an `if` inside a `for` loop.

**What the check measures:** Your `score_field` (Step 2) on new data, `CAUSE_LINE` (Step 4) and `KEEP` (Step 5).

**Time:** about 20 minutes.

**Done when:** the last cell prints `Done:`.

## The situation

A small online shop sends each customer message, with a prompt, to an AI model. The model extracts four fields as JSON: `order_id`, `issue` (late, damaged, wrong_item or other), `refund_amount` (the money asked back, or null if none is named; None in Python) and `urgent` (true or false).

A teammate read a few v1 replies; they looked fine. You will score prompt v1 on a labelled set: 8 messages, each with its label. All data here is simulated.

## Step 1 · Look before you measure

Run the cell.

Expect three messages, each with its label and v1 reply. Find the field where they differ.

<details><summary>Hint</summary>

Compare the values, key by key.

</details>

<details><summary>Show the answer</summary>

`refund_amount` in message 2: the reply gives 0, but no amount is named, so the label has None.

</details>

In [ ]:
# Step 1: load the labelled set and v1 replies; show three
FIELDS = ["order_id", "issue", "refund_amount", "urgent"]
MESSAGES = [
    "Order A1042 came late. Refund my $5 delivery fee.",
    "Order A1187: the mug arrived broken. Send another.",
    "Order A1210 is the wrong size. Refund my $24 today.",
    "Order A1305 is two weeks late. Where is it?",
    "Order A1456 has someone else's shoes. I need mine today!",
    "Order A1501 cost $60. Refund just the $8 gift wrap; it never came.",
    "Can I change the colour of order A1533?",
    "Order A1609 came late, and the vase was smashed. Refund its $15.",
]
# Labels, checked by a person
LABELS = [
    dict(order_id="A1042", issue="late", refund_amount=5, urgent=False),
    dict(order_id="A1187", issue="damaged", refund_amount=None, urgent=False),
    dict(order_id="A1210", issue="wrong_item", refund_amount=24, urgent=True),
    dict(order_id="A1305", issue="late", refund_amount=None, urgent=False),
    dict(order_id="A1456", issue="wrong_item", refund_amount=None, urgent=True),
    dict(order_id="A1501", issue="other", refund_amount=8, urgent=False),
    dict(order_id="A1533", issue="other", refund_amount=None, urgent=False),
    dict(order_id="A1609", issue="damaged", refund_amount=15, urgent=False),
]
# Sample replies (simulated) to prompt v1
REPLIES_V1 = [
    dict(order_id="A1042", issue="late", refund_amount=5, urgent=False),
    dict(order_id="A1187", issue="damaged", refund_amount=0, urgent=False),
    dict(order_id="A1210", issue="wrong_item", refund_amount=24, urgent=True),
    dict(order_id="A1305", issue="late", refund_amount=0, urgent=False),
    dict(order_id="A1456", issue="wrong_item", refund_amount=0, urgent=True),
    dict(order_id="A1501", issue="other", refund_amount=8, urgent=False),
    dict(order_id="A1533", issue="other", refund_amount=0, urgent=False),
    dict(order_id="A1609", issue="late", refund_amount=15, urgent=False),
]
for i in range(3):
    print(f"Message {i + 1}: {MESSAGES[i]}")
    print("  label:", LABELS[i])
    print("  reply:", REPLIES_V1[i])

## Step 2 · Score one field

`score_field`, your scorer, counts, for one field:

- **expected**: labels with a value; already done.
- **given**: replies with a value. 0 and False are values; only None is not.
- **correct**: given values equal to the label.

**Precision** = correct / given: the share of given values that were right. **Recall** = correct / expected: the share of expected values that were found.

Do each TODO, then run the cell.

Expect `{'correct': 2, 'given': 3, 'expected': 2, 'precision': 0.6666666666666666, 'recall': 1.0}`, then precision and recall 1.0 for perfect replies.

<details><summary>Hint</summary>

Copy the `expected` lines, using `reply` and `given`. Inside, test `reply[field] == label[field]`.

</details>

<details><summary>Show the answer</summary>

```python
        if reply[field] is not None:
            given += 1
            if reply[field] == label[field]:
                correct += 1
    precision = correct / given
    recall = correct / expected
```

`is not None` counts the invented 0; `if reply[field]:` would skip it.

</details>

In [ ]:
# Step 2: score one field; try it on Step 1's messages and perfect replies
def score_field(field: str, labels: list[dict], replies: list[dict]) -> dict:
    """Count one field's values; work out its precision and recall."""
    correct, given, expected = 0, 0, 0
    for label, reply in zip(labels, replies):  # each label with its reply
        if label[field] is not None:
            expected += 1
        # TODO: count given, and inside it, correct.
    precision = None  # TODO: correct / given
    recall = None  # TODO: correct / expected
    return dict(correct=correct, given=given, expected=expected,
                precision=precision, recall=recall)


print(score_field("refund_amount", LABELS[:3], REPLIES_V1[:3]))
print(score_field("refund_amount", LABELS, LABELS))  # perfect replies: the labels

## Step 3 · Score prompt v1 against the bar

The bar is the score each field must reach: at least 0.8, for precision and recall. Run the cell.

Expect `refund_amount` BELOW THE BAR: precision 0.50, recall 1.00. Invented zeros lower precision, not recall. `issue` shows 0.88: message 8, not shown, asks $15 back for a smashed vase; the reply says late, not damaged.

In [ ]:
# Step 3: score the v1 replies against the bar
BAR = 0.8


def report(replies: list[dict]) -> None:
    """Print each field's precision, recall and verdict."""
    for field in FIELDS:
        s = score_field(field, LABELS, replies)
        if None in (s["precision"], s["recall"]):
            print(field, "is not scored yet: finish Step 2")
            continue
        lower = min(s["precision"], s["recall"])
        verdict = "meets the bar" if lower >= BAR else "BELOW THE BAR"
        print(f"{field:14} precision {s['precision']:.2f} ({s['correct']}/{s['given']})  "
              f"recall {s['recall']:.2f} ({s['correct']}/{s['expected']})  {verdict}")


report(REPLIES_V1)

## Step 4 · Find the cause

A failure is what went wrong; its cause is why, and it points at the fix (Slide 30). Run the cell.

Expect prompt v1, its lines numbered. Set `CAUSE_LINE` to the number of the line that makes the model write 0, then run the cell again.

<details><summary>Hint</summary>

Line 3 asks for null. Models copy examples.

</details>

<details><summary>Show the answer</summary>

`CAUSE_LINE = 6`. Line 5's message names no amount, yet line 6's example reply gives `"refund_amount": 0`, and the model copies it.

</details>

In [ ]:
# Step 4: show prompt v1 with its line numbers
PROMPT_V1 = """Read the customer's message. Reply with JSON only, with four keys.
order_id: the order number. issue: late, damaged, wrong_item or other.
refund_amount: the money asked back, or null if the message names no amount.
urgent: true if help is needed today, else false.
Example message: Order A2001 came late. No rush.
Example reply: {"order_id": "A2001", "issue": "late", "refund_amount": 0, "urgent": false}
"""
for number, line in enumerate(PROMPT_V1.splitlines(), start=1):
    print(number, line)

# TODO: set CAUSE_LINE to a line number.
CAUSE_LINE = None

## Step 5 · Score prompt v2 and choose

Prompt v2 fixes that line. The same labels, scorer and bar judge it. Run the cell.

Expect every field to meet the bar: `refund_amount` rises to 1.00; `urgent`, untouched by the change, falls to 0.88 (message 5). Set `KEEP` to `"v1"` or `"v2"`, whichever meets the bar on every field, then run the cell again.

<details><summary>Hint</summary>

Compare Step 3's verdicts with these.

</details>

<details><summary>Show the answer</summary>

`KEEP = "v2"`: v1 fails on `refund_amount`; v2 meets the bar on every field.

</details>

In [ ]:
# Step 5: make prompt v2, then score its replies
PROMPT_V2 = PROMPT_V1.replace('"refund_amount": 0', '"refund_amount": null')
# Sample replies (simulated) to prompt v2
REPLIES_V2 = [
    dict(order_id="A1042", issue="late", refund_amount=5, urgent=False),
    dict(order_id="A1187", issue="damaged", refund_amount=None, urgent=False),
    dict(order_id="A1210", issue="wrong_item", refund_amount=24, urgent=True),
    dict(order_id="A1305", issue="late", refund_amount=None, urgent=False),
    dict(order_id="A1456", issue="wrong_item", refund_amount=None, urgent=False),
    dict(order_id="A1501", issue="other", refund_amount=8, urgent=False),
    dict(order_id="A1533", issue="other", refund_amount=None, urgent=False),
    dict(order_id="A1609", issue="late", refund_amount=15, urgent=False),
]
report(REPLIES_V2)

# TODO: set KEEP to "v1" or "v2".
KEEP = None

## Check your work

Run the cell. Expect a line that starts with `Done:`.

In [ ]:
# Check your work: test score_field, CAUSE_LINE and KEEP
RIGHT = dict(given=4, expected=2, correct=1, precision=0.25, recall=0.5)
problem = ""
try:
    got = score_field("x", [dict(x=v) for v in (None, 5, 3, None, None)],
                      [dict(x=v) for v in (0, 5, 4, 9, None)])
    bad = [key for key in RIGHT if got.get(key) != RIGHT[key]]
    if bad:
        problem = (f"Step 2: on new data, {bad[0]} is {got.get(bad[0])}, "
                   f"not {RIGHT[bad[0]]}; fix score_field")
except Exception as error:
    problem = f"Step 2: score_field stops with {type(error).__name__}: {error}; fix it"
if not problem and str(globals().get("CAUSE_LINE")).strip() != "6":
    problem = "Step 4: set CAUSE_LINE to a number: the line whose example reply writes 0"
if not problem and str(globals().get("KEEP")).strip().lower() != "v2":
    problem = 'Step 5: set KEEP to "v1" or "v2", whichever meets the bar on every field'
print(f"Not yet: {problem}, then run that step's cell and this one." if problem else
      "Done: you scored both prompts, traced the invented 0 to line 6 and kept v2.")

## What you learned

- By eye, three replies showed one wrong field; the labelled set showed how often, and a wrong issue you never saw (Slides 22 and 23).
- An invented value lowers precision; a missed one lowers recall (Slide 27).
- Rescore every field on every change, with the same labels, scorer and bar (Slide 24).

**Reflect:** A message asks for $20 back, and the reply gives null. Which measure drops, and why?